# Qwen3-0.6B (heretic / abliterated) on vLLM — Kaggle

Runs [`DavidAU/Qwen3-0.6B-heretic-abliterated-uncensored`](https://huggingface.co/DavidAU/Qwen3-0.6B-heretic-abliterated-uncensored) (0.6B parameters) with **vLLM** as a local OpenAI-compatible server.

**Before you run (Notebook settings, right sidebar):**
1. **Accelerator:** `GPU T4 x2` (only one GPU is used). *P100 does not work — vLLM needs compute capability 7.0+.*
2. **Internet:** `On` (needed to download vLLM and the model; Kaggle requires a verified phone number for this).
3. Then **Run All**. First run takes roughly 5–10 minutes (mostly the vLLM install).

**Why it is built this way:** vLLM ships its own PyTorch, which would clash with Kaggle's preinstalled one. So vLLM is installed into an isolated virtual environment (`/tmp/vllm-env`) and run as a background server. The notebook kernel never imports vLLM, so there is no kernel restart and Kaggle's own packages stay untouched.

## 1. Check the GPU

In [ ]:
import subprocess

def check_gpu():
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    if r.returncode != 0 or not r.stdout.strip():
        raise RuntimeError("No GPU detected. Set Accelerator to 'GPU T4 x2' in the notebook settings and re-run.")
    lines = [l.strip() for l in r.stdout.strip().splitlines()]
    for i, l in enumerate(lines):
        print(f"GPU {i}: {l}")
    name, mem, cap = [x.strip() for x in lines[0].split(",")]
    if float(cap) < 7.0:
        raise RuntimeError(
            f"GPU 0 ({name}) has compute capability {cap}. vLLM needs 7.0 or higher. "
            "Choose 'GPU T4 x2' instead of 'GPU P100'."
        )
    print("GPU check passed.")

check_gpu()

## 2. Configuration and helpers

In [ ]:
import os, sys, time, json, signal, subprocess, urllib.request

# ---- model / server settings (edit here) ----
MODEL_ID      = "DavidAU/Qwen3-0.6B-heretic-abliterated-uncensored"
SERVED_NAME   = "qwen3-0.6b-heretic"
PORT          = 8000
MAX_MODEL_LEN = 4096      # model supports ~40k; keep small for a quick test
GPU_MEM_UTIL  = 0.80      # fraction of ONE T4 (15 GB) that vLLM may reserve
DTYPE         = "float16" # T4 has no bfloat16 support
ENFORCE_EAGER = True      # True = skip CUDA-graph capture: faster start, fewer failure modes
VLLM_SPEC     = "vllm"    # pin if you like, e.g. "vllm==0.11.0"

# ---- paths ----
VENV_DIR = "/tmp/vllm-env"        # /tmp so it is NOT saved into /kaggle/working output
HF_HOME  = "/tmp/hf"
LOG_FILE = "/tmp/vllm_server.log"


def clean_env(extra=None):
    """Environment for the isolated venv: drop Kaggle's python/CUDA paths so they cannot leak in."""
    env = os.environ.copy()
    env.pop("PYTHONPATH", None)
    env.pop("PYTHONHOME", None)
    keep = [p for p in env.get("LD_LIBRARY_PATH", "").split(":")
            if p.startswith("/usr/local/nvidia") or p.startswith("/usr/lib")]
    if keep:
        env["LD_LIBRARY_PATH"] = ":".join(keep)
    else:
        env.pop("LD_LIBRARY_PATH", None)
    env["HF_HOME"] = HF_HOME
    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["UV_HTTP_TIMEOUT"] = "600"
    if extra:
        env.update(extra)
    return env


def run(cmd, **kw):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, **kw)


print("Config OK ->", MODEL_ID)

## 3. Install vLLM into an isolated environment
Downloads several GB (vLLM + PyTorch). Skipped automatically on re-run if already installed.

In [ ]:
VENV_PY = f"{VENV_DIR}/bin/python"

run([sys.executable, "-m", "pip", "install", "-q", "uv"])

if not os.path.exists(VENV_PY):
    run([sys.executable, "-m", "uv", "venv", VENV_DIR, "--python", sys.executable], env=clean_env())

probe = subprocess.run([VENV_PY, "-c", "import vllm"], capture_output=True, env=clean_env())
if probe.returncode != 0:
    run([sys.executable, "-m", "uv", "pip", "install", "--python", VENV_PY, VLLM_SPEC], env=clean_env())
else:
    print("vLLM already installed in the venv, skipping install.")

check = subprocess.run(
    [VENV_PY, "-c",
     "import vllm, torch; "
     "print('vllm', vllm.__version__); "
     "print('torch', torch.__version__, 'cuda', torch.version.cuda); "
     "print('cuda_available', torch.cuda.is_available())"],
    capture_output=True, text=True, env=clean_env(),
)
print(check.stdout)
if check.returncode != 0 or "cuda_available True" not in check.stdout:
    print(check.stderr[-3000:])
    raise RuntimeError("vLLM/PyTorch in the venv cannot see the GPU. See the output above.")
print("Install verified.")

## 4. Start the vLLM server
Runs in the background and logs to `/tmp/vllm_server.log`. The first start downloads the model (~1.2 GB) and warms up the engine.

In [ ]:
server = globals().get("server", None)


def tail_log(n=80):
    try:
        with open(LOG_FILE, "r", errors="replace") as f:
            return "".join(f.readlines()[-n:])
    except FileNotFoundError:
        return "(no log file yet)"


def wait_until_ready(timeout=1200):
    t0 = last_note = time.time()
    while time.time() - t0 < timeout:
        if server.poll() is not None:
            print(tail_log(100))
            raise RuntimeError(f"vLLM server exited early (code {server.returncode}). Log tail printed above.")
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2) as r:
                if r.status == 200:
                    print(f"Server ready after {time.time() - t0:.0f}s")
                    return
        except Exception:
            pass
        if time.time() - last_note > 30:
            print(f"  ...still starting ({time.time() - t0:.0f}s)")
            last_note = time.time()
        time.sleep(3)
    print(tail_log(100))
    raise TimeoutError("vLLM server did not become ready in time. Log tail printed above.")


if server is not None and server.poll() is None:
    print("Server is already running.")
else:
    cmd = [
        f"{VENV_DIR}/bin/vllm", "serve", MODEL_ID,
        "--served-model-name", SERVED_NAME,
        "--host", "127.0.0.1",
        "--port", str(PORT),
        "--dtype", DTYPE,
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_MEM_UTIL),
        "--tensor-parallel-size", "1",
    ]
    if ENFORCE_EAGER:
        cmd.append("--enforce-eager")

    extra = {}
    try:  # optional: a Kaggle secret named HF_TOKEN (not needed for this public model)
        from kaggle_secrets import UserSecretsClient
        extra["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

    print("$", " ".join(cmd))
    log = open(LOG_FILE, "w")
    server = subprocess.Popen(
        cmd, stdout=log, stderr=subprocess.STDOUT,
        env=clean_env(extra), start_new_session=True,
    )
    wait_until_ready()

with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=10) as r:
    print("Serving:", [m["id"] for m in json.loads(r.read())["data"]])

## 5. Chat helper
Calls the server's OpenAI-compatible `/v1/chat/completions` endpoint. Qwen3 has a "thinking" mode; it is **off** by default here to keep answers short — pass `thinking=True` to enable it.

In [ ]:
import requests

BASE = f"http://127.0.0.1:{PORT}/v1"


def chat(prompt, system=None, max_tokens=512, temperature=0.7, top_p=0.9, thinking=False):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    payload = {
        "model": SERVED_NAME,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": top_p,
        "chat_template_kwargs": {"enable_thinking": thinking},
    }
    r = requests.post(f"{BASE}/chat/completions", json=payload, timeout=300)
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")
    data = r.json()
    return data["choices"][0]["message"]["content"], data["usage"]


text, usage = chat("In three sentences, explain what vLLM is and why it is fast.")
print(text)
print("\nusage:", usage)

## 6. Throughput test (vLLM batches concurrent requests)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

prompts = [f"Write one short fun fact about the number {i}." for i in range(1, 33)]

t0 = time.time()
with ThreadPoolExecutor(max_workers=32) as pool:
    results = list(pool.map(lambda p: chat(p, max_tokens=64), prompts))
elapsed = time.time() - t0

gen_tokens = sum(u["completion_tokens"] for _, u in results)
print(f"{len(prompts)} requests in {elapsed:.1f}s  |  {gen_tokens} generated tokens  |  {gen_tokens / elapsed:.0f} tokens/s")
print("\nSample:", results[0][0])

## 7. Try your own prompt

In [ ]:
your_prompt = "Write a short, dark two-sentence story set in a lighthouse."

text, _ = chat(your_prompt, temperature=0.8, max_tokens=300)
print(text)

## 8. Server logs (use if something goes wrong)

In [ ]:
print(tail_log(60))

## 9. Stop the server (frees the GPU)

In [ ]:
def stop_server():
    global server
    if server is not None and server.poll() is None:
        pgid = os.getpgid(server.pid)
        os.killpg(pgid, signal.SIGTERM)
        try:
            server.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(pgid, signal.SIGKILL)
            server.wait()
        print("Server stopped.")
    else:
        print("Server is not running.")

stop_server()